# 3.2 Preprocessing

## 3.2.0 Train-test split
We split the training and testing set according to date, we split the data before 2026-05-25 as training set, the data after that as testing set.

In [7]:
import duckdb
from pathlib import Path

# import library and connect to db
con = duckdb.connect()

In [8]:
# load dataset
data_path = Path("../data/labeled.parquet")

df = con.sql(f"""
    SELECT *
    FROM read_parquet('{data_path}')
""")

df.limit(5).df()

,date,serial_number,model,capacity_bytes,failure,datacenter,cluster_id,vault_id,pod_id,pod_slot_num,...,smart_240_normalized,smart_240_raw,smart_241_normalized,smart_241_raw,smart_242_normalized,smart_242_raw,days_to_failure,label_3d,label_7d,label_30d
0,2026-01-01,1030A00UF97G,TOSHIBA MG07ACA14TA,14000519643136,0,sac2,000,1108,09,0013,...,100.0,0.0,NaN,NaN,NaN,NaN,<NA>,0,0,0
1,2026-01-02,1030A00UF97G,TOSHIBA MG07ACA14TA,14000519643136,0,sac2,000,1108,09,0013,...,100.0,0.0,NaN,NaN,NaN,NaN,<NA>,0,0,0
2,2026-01-03,1030A00UF97G,TOSHIBA MG07ACA14TA,14000519643136,0,sac2,000,1108,09,0013,...,100.0,0.0,NaN,NaN,NaN,NaN,<NA>,0,0,0
3,2026-01-04,1030A00UF97G,TOSHIBA MG07ACA14TA,14000519643136,0,sac2,000,1108,09,0013,...,100.0,0.0,NaN,NaN,NaN,NaN,<NA>,0,0,0
4,2026-01-05,1030A00UF97G,TOSHIBA MG07ACA14TA,14000519643136,0,sac2,000,1108,09,0013,...,100.0,0.0,NaN,NaN,NaN,NaN,<NA>,0,0,0


### Temporal Train-Test Split

In [9]:
cutoff = "2026-05-25"

train = con.sql(f"""
    SELECT *
    FROM read_parquet('{data_path}')
    WHERE date < DATE '{cutoff}'
""")

test = con.sql(f"""
    SELECT *
    FROM read_parquet('{data_path}')
    WHERE date >= DATE '{cutoff}'
""")

### verify split

In [10]:
print("Training Set:")
print(train.aggregate("COUNT(*) AS n, MIN(date) AS start, MAX(date) AS end").df())

print("\nTesting Set:")
print(test.aggregate("COUNT(*) AS n, MIN(date) AS start, MAX(date) AS end").df())

Training Set:
         n      start        end
0  1882420 2026-01-01 2026-05-24

Testing Set:
        n      start        end
0  479223 2026-05-25 2026-06-30


### Save training and testing set

In [11]:
con.sql(f"""
    COPY (
        SELECT * FROM read_parquet('{data_path}')
        WHERE date < DATE '{cutoff}'
    )
    TO '../data/train.parquet' (FORMAT PARQUET)
""")

con.sql(f"""
    COPY (
        SELECT * FROM read_parquet('{data_path}')
        WHERE date >= DATE '{cutoff}'
    )
    TO '../data/test.parquet' (FORMAT PARQUET)
""")

CountBIGINT
479223


## 3.2.1 Handling Missing Value
In 3.1.3 data quality, we already removed features carry no information (no value for all three models). In this part, we carry out Model-wise median imputation, and provide fixed placeholder values ​​used for SMART attributes that are entirely unavailable for a given model (Structural Missingness).

### Calculate Median 
Use the training set to calculate the median for each SMART feature, grouped by hard drive model.

In [12]:
train_path = "../data/train.parquet"
test_path = "../data/test.parquet"

smart_cols = [
    col for col in train.columns
    if col.startswith("smart_")
]

median_expr = ",\n".join(
    f'median("{col}") AS "{col}"'
    for col in smart_cols
)

con.sql(f"""
    CREATE OR REPLACE TEMP TABLE train_medians AS
    SELECT
        model,
        {median_expr}
    FROM '{train_path}'
    GROUP BY model
""")

CountBIGINT
3


### Impute Missing Values
Use the median to impute both training and testing set.

In [13]:
impute_expr = ",\n".join(
    f'COALESCE(t."{col}", m."{col}", 0) AS "{col}"'
    for col in smart_cols
)

def impute_dataset(input_path, output_path):
    con.sql(f"""
        COPY (
            SELECT t.* REPLACE (
                {impute_expr}
            )
            FROM '{input_path}' AS t
            LEFT JOIN train_medians AS m
                ON t.model = m.model
        )
        TO '{output_path}' (FORMAT PARQUET)
    """)

impute_dataset(
    train_path,
    "../data/train_imputed.parquet"
)

impute_dataset(
    test_path,
    "../data/test_imputed.parquet"
)

### Verify Imputation
Verify whether there still any exist missing values.

In [15]:

for dataset in ["train_imputed", "test_imputed"]:
    path = f"../data/{dataset}.parquet"

    missing_expr = " + ".join(
        f'COUNT(*) FILTER (WHERE "{col}" IS NULL)'
        for col in smart_cols
    )

    missing_count = con.execute(f"""
        SELECT {missing_expr}
        FROM '{path}'
    """).fetchone()[0]

    print(f"{dataset}: {missing_count} missing values")


train_imputed: 0 missing values
test_imputed: 0 missing values


## 3.2.2 Normalization and Encoding


For normalizing numerical values, we will normalize the SMART values. We need to consider that we have two different forms of SMART values in our dataset, `smart_x_raw` and `smart_x_normalized`. `smart_x_raw` represents the raw values ​​reported by a hard drive, such as cumulative error counts, total operating hours, temperature, etc. `smart_x_normalized` stands for Values ​​that have been processed by the hard drive manufacturer. We carry out Z-score standardation on `smart_x_raw` and `smart_x_normalized` seperately. Moreover, in 3.1.5 5. Univariate analysis, we found that `smart_5_raw`, `smart_187_raw`, `smart_188_raw`, `smart_197_raw`, and `smart_198_raw` are right-skewed. In this part, we will also use log transformation on these features.

### Normalization

In [16]:
log_cols = {
    "smart_5_raw",
    "smart_187_raw",
    "smart_188_raw",
    "smart_197_raw",
    "smart_198_raw"
}

def feature_expr(alias, col):
    x = f'{alias}."{col}"'
    return f"LN(1.0 + {x})" if col in log_cols else x

# Calculate mean and std using training data only
stats_expr = ",\n".join(
    f'AVG({feature_expr("t", col)}) AS "{col}_mean", '
    f'STDDEV_POP({feature_expr("t", col)}) AS "{col}_std"'
    for col in smart_cols
)

con.execute(f"""
    CREATE OR REPLACE TEMP TABLE smart_scale_stats AS
    SELECT {stats_expr}
    FROM '../data/train_imputed.parquet' AS t
""")


scale_expr = ",\n".join(
    f'''CASE
            WHEN s."{col}_std" > 0
            THEN ({feature_expr("t", col)} - s."{col}_mean")
                 / s."{col}_std"
            ELSE 0.0
        END AS "{col}"'''
    for col in smart_cols
)

def standardize_dataset(input_path, output_path):
    con.execute(f"""
        COPY (
            SELECT t.* REPLACE (
                {scale_expr}
            )
            FROM '{input_path}' AS t
            CROSS JOIN smart_scale_stats AS s
        )
        TO '{output_path}' (FORMAT PARQUET)
    """)

standardize_dataset(
    "../data/train_imputed.parquet",
    "../data/train_scaled.parquet"
)

standardize_dataset(
    "../data/test_imputed.parquet",
    "../data/test_scaled.parquet"
)


### Verify Normalization

In [17]:
cols_to_check = [
    "smart_5_raw",
    "smart_5_normalized",
    "smart_194_raw",
    "smart_194_normalized"
]

check_expr = ",\n".join(
    f'ROUND(AVG("{col}"), 4) AS "{col}_mean", '
    f'ROUND(STDDEV_POP("{col}"), 4) AS "{col}_std"'
    for col in cols_to_check
)

con.sql(f"""
    SELECT {check_expr}
    FROM '../data/train_scaled.parquet'
""").df()


,smart_5_raw_mean,smart_5_raw_std,smart_5_normalized_mean,smart_5_normalized_std,smart_194_raw_mean,smart_194_raw_std,smart_194_normalized_mean,smart_194_normalized_std
0,0.0,1.0,0.0,1.0,0.0,1.0,-0.0,1.0


## One-hot Encoding
We do a one-hot encoding on `model` attribute. It contains three different drive models, `model_ST12000NM0008`, `model_TOSHIBA_MG07ACA14TA`, `model_HGST_HUH721212ALN604`.

In [18]:
import re

models = [
    row[0]
    for row in con.execute("""
        SELECT DISTINCT model
        FROM '../data/train_scaled.parquet'
        WHERE model IS NOT NULL
        ORDER BY model
    """).fetchall()
]

print("Models:", models)

# Generate one-hot encoding expressions
model_columns = {
    model: "model_" + re.sub(r"\W+", "_", model)
    for model in models
}

encoding_expr = ",\n".join(
    f"""CASE WHEN model = '{model.replace("'", "''")}'
        THEN 1 ELSE 0 END AS "{column}" """
    for model, column in model_columns.items()
)

def encode_dataset(input_path, output_path):
    con.execute(f"""
        COPY (
            SELECT
                * EXCLUDE (model),
                {encoding_expr}
            FROM '{input_path}'
        )
        TO '{output_path}' (FORMAT PARQUET)
    """)

encode_dataset(
    "../data/train_scaled.parquet",
    "../data/train_encoded.parquet"
)

encode_dataset(
    "../data/test_scaled.parquet",
    "../data/test_encoded.parquet"
)


Models: ['HGST HUH721212ALN604', 'ST12000NM0008', 'TOSHIBA MG07ACA14TA']


### Verify One-hot encoding

In [ ]:
encoded_cols = list(model_columns.values())
col_expr = ", ".join(f'"{col}"' for col in encoded_cols)

for dataset in ["train_encoded", "test_encoded"]:
    print(f"\n{dataset}:")

    result = con.sql(f"""
        SELECT
            {col_expr},
            COUNT(*) AS records
        FROM '../data/{dataset}.parquet'
        GROUP BY {col_expr}
        ORDER BY records DESC
    """).df()


    display(result)



train_encoded:


,model_HGST_HUH721212ALN604,model_ST12000NM0008,model_TOSHIBA_MG07ACA14TA,records
0,0,0,1,1069051
1,0,1,0,531131
2,1,0,0,282238



test_encoded:


,model_HGST_HUH721212ALN604,model_ST12000NM0008,model_TOSHIBA_MG07ACA14TA,records
0,0,0,1,273384
1,0,1,0,135254
2,1,0,0,70585
